# Automated Substation Creation and Feeder Insertion in CIM18

This notebook builds each node-breaker substation layout supported by CIM-Builder around the IEEE 13
and IEEE 13 Assets feeders. For each layout it:

1. loads both feeders into a fresh CIMantic Graphs model,
2. builds the substation, its 115 kV VoltageLevel, buses and bus ties,
3. adds a breaker bay from the bus to a new 115 kV line node and to each feeder's `sourcebus`,
4. links both feeders to the substation through `Feeder.NormalEnergizingSubstation`,
5. writes the substation and feeders to one CIM XML file, then reads it back to check the
   substation, its switches and the total load served.

The per-layout notebooks walk through each of these steps in detail.

----

### Import CIMantic Graphs, the CIM profile and the builders

In [1]:
import logging
import os

os.environ["CIMG_CIM_PROFILE"] = "cim18v17"
os.environ["CIMG_USE_UNITS"] = "TRUE"
os.environ["CIMG_SYSTEM_MVA_BASE"] = "100"
# The IEEE 13 files carry CIMHub extensions that cim18v17 does not have: hide the parser warnings
logging.getLogger("cimgraph").setLevel(logging.ERROR)

import cimgraph.utils as utils
import cimprofiles.cim18.cim18v17 as cim
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel
from cimtables.units import Qty

from cimbuilder.object_builder import BaseVoltageBuilder, FeederBuilder
from cimbuilder.substation_builder import (
    BreakerAndHalfSubstationBuilder,
    DoubleBusSingleBreakerSubstationBuilder,
    MainAndTransferSubstationBuilder,
    RingBusSubstationBuilder,
    SectionalizedBusSubstationBuilder,
    SingleBusSubstationBuilder,
)

### Shared helpers

These load the feeders, find each feeder's head node, link the feeders, and save and check the result. The layout sections below differ only in their builder and their `add_buses` / `add_branch` arguments.

In [2]:
IEEE13 = "49AD8E07-3BF9-A4E2-CB8F-C3722F837B62"
IEEE13_ASSETS = "5B816B93-7A5F-B64C-8460-47C17D6E4B0F"
FEEDER_FILES = ["../../tests/test_models/IEEE13.xml", "../../tests/test_models/IEEE13_Assets.xml"]


def load_feeders():
    """a fresh graph of both IEEE 13 feeders and a 115 kV BaseVoltage, bv_115kV, for the substation"""
    network = FeederModel(
        container=cim.Feeder(mRID=IEEE13), connection=XMLFile(filename=FEEDER_FILES), distributed=False
    )
    BaseVoltageBuilder(network, network.container).create(
        mRID="bv_115kV", name="115 kV", nominalVoltage=Qty(115, "kV")
    )
    return network


def sourcebus(network, feeder_mRID):
    """mRID of the feeder head node, the ConnectivityNode named sourcebus in the feeder"""
    for node in network.graph[cim.ConnectivityNode].values():
        node_container = node.ConnectivityNodeContainer
        if node.name == "sourcebus" and node_container is not None and node_container.mRID == feeder_mRID:
            return node.mRID
    raise ValueError(f"no sourcebus in feeder {feeder_mRID}")


def link_feeders(network, substation):
    """Feeder.NormalEnergizingSubstation of both feeders"""
    feeder_builder = FeederBuilder(network, network.container)
    for feeder_mRID in (IEEE13, IEEE13_ASSETS):
        feeder_builder.add_connectivity(
            network.find_by_mRID(feeder_mRID), NormalEnergizingSubstation=substation.mRID
        )


def save_and_check(network, substation, filename):
    """writes the network to output/<filename>, reads it back and prints what came back"""
    os.makedirs("output", exist_ok=True)
    utils.write_xml(network, f"output/{filename}")
    roundtrip = FeederModel(
        container=cim.Feeder(mRID=IEEE13), connection=XMLFile(filename=f"output/{filename}"), distributed=False
    )
    substation = roundtrip.find_by_mRID(substation.mRID)
    switches = [
        equipment
        for voltage_level in substation.VoltageLevels
        for equipment in voltage_level.Equipments
        if isinstance(equipment, (cim.Breaker, cim.Disconnector))
    ]
    total_load = sum(float(load.p) for load in roundtrip.graph[cim.EnergyConsumer].values())
    print(f"{substation.mRID}: feeders {[feeder.name for feeder in substation.NormalEnergizedFeeder]}")
    print(f"  {len(switches)} breakers and disconnectors in the substation")
    print(f"  total load served: {cim.ActivePower(total_load).to('kW'):.1f} kW")

## Single Bus

One bus. Each branch is a breaker bay off the bus.

In [3]:
network = load_feeders()
builder = SingleBusSubstationBuilder(network, network.container)
substation = builder.create(mRID="sub_single_bus", name="sub_single_bus")
substation, voltage_level = builder.add_voltage_level(substation, mRID="vl_115", BaseVoltage="bv_115kV")
voltage_level, busbars, ties = builder.add_buses(voltage_level)
builder.add_branch(voltage_level, number=1, node="line_1")
builder.add_branch(voltage_level, number=2, node=sourcebus(network, IEEE13))
builder.add_branch(voltage_level, number=3, node=sourcebus(network, IEEE13_ASSETS))
link_feeders(network, substation)
save_and_check(network, substation, "single_bus.xml")

sub_single_bus: feeders ['ieee13nodeckt', 'ieee13nodecktassets']
  9 breakers and disconnectors in the substation
  total load served: 6937.0 kW


## Sectionalized Bus

Two bus sections joined by a tie bay. `section` picks the section each branch is on.

In [4]:
network = load_feeders()
builder = SectionalizedBusSubstationBuilder(network, network.container)
substation = builder.create(mRID="sub_sectionalized", name="sub_sectionalized")
substation, voltage_level = builder.add_voltage_level(substation, mRID="vl_115", BaseVoltage="bv_115kV")
voltage_level, busbars, ties = builder.add_buses(voltage_level, sections=2)
builder.add_branch(voltage_level, section=1, number=1, node="line_1")
builder.add_branch(voltage_level, section=1, number=2, node=sourcebus(network, IEEE13))
builder.add_branch(voltage_level, section=2, number=3, node=sourcebus(network, IEEE13_ASSETS))
link_feeders(network, substation)
save_and_check(network, substation, "sectionalized_bus.xml")

sub_sectionalized: feeders ['ieee13nodeckt', 'ieee13nodecktassets']
  12 breakers and disconnectors in the substation
  total load served: 6937.0 kW


## Main and Transfer

A main bus and a transfer bus joined by a tie bay. Each branch also gets a normally-open bypass disconnector to the transfer bus.

In [5]:
network = load_feeders()
builder = MainAndTransferSubstationBuilder(network, network.container)
substation = builder.create(mRID="sub_main_and_transfer", name="sub_main_and_transfer")
substation, voltage_level = builder.add_voltage_level(substation, mRID="vl_115", BaseVoltage="bv_115kV")
voltage_level, busbars, ties = builder.add_buses(voltage_level)
builder.add_branch(voltage_level, number=1, node="line_1")
builder.add_branch(voltage_level, number=2, node=sourcebus(network, IEEE13))
builder.add_branch(voltage_level, number=3, node=sourcebus(network, IEEE13_ASSETS))
link_feeders(network, substation)
save_and_check(network, substation, "main_and_transfer.xml")

sub_main_and_transfer: feeders ['ieee13nodeckt', 'ieee13nodecktassets']
  15 breakers and disconnectors in the substation
  total load served: 6937.0 kW


## Double Bus Single Breaker

Two buses joined by a tie bay. `bus` picks the bus each branch normally runs on; the selector to the other bus is normally open.

In [6]:
network = load_feeders()
builder = DoubleBusSingleBreakerSubstationBuilder(network, network.container)
substation = builder.create(mRID="sub_double_bus", name="sub_double_bus")
substation, voltage_level = builder.add_voltage_level(substation, mRID="vl_115", BaseVoltage="bv_115kV")
voltage_level, busbars, ties = builder.add_buses(voltage_level)
builder.add_branch(voltage_level, number=1, node="line_1", bus=1)
builder.add_branch(voltage_level, number=2, node=sourcebus(network, IEEE13), bus=1)
builder.add_branch(voltage_level, number=3, node=sourcebus(network, IEEE13_ASSETS), bus=2)
link_feeders(network, substation)
save_and_check(network, substation, "double_bus_single_breaker.xml")

sub_double_bus: feeders ['ieee13nodeckt', 'ieee13nodecktassets']
  15 breakers and disconnectors in the substation
  total load served: 6937.0 kW


## Ring Bus

Four sections in a ring of breaker bays. Each branch connects to its section through a disconnector.

In [7]:
network = load_feeders()
builder = RingBusSubstationBuilder(network, network.container)
substation = builder.create(mRID="sub_ring_bus", name="sub_ring_bus")
substation, voltage_level = builder.add_voltage_level(substation, mRID="vl_115", BaseVoltage="bv_115kV")
voltage_level, busbars, ring = builder.add_buses(voltage_level, sections=4)
builder.add_branch(voltage_level, section=1, node="line_1")
builder.add_branch(voltage_level, section=3, node=sourcebus(network, IEEE13))
builder.add_branch(voltage_level, section=4, node=sourcebus(network, IEEE13_ASSETS))
link_feeders(network, substation)
save_and_check(network, substation, "ring_bus.xml")

sub_ring_bus: feeders ['ieee13nodeckt', 'ieee13nodecktassets']
  15 breakers and disconnectors in the substation
  total load served: 6937.0 kW


## Breaker and a Half

Two buses joined by two diameters of three breaker bays each. A branch connects to one side of a diameter.

In [8]:
network = load_feeders()
builder = BreakerAndHalfSubstationBuilder(network, network.container)
substation = builder.create(mRID="sub_breaker_and_half", name="sub_breaker_and_half")
substation, voltage_level = builder.add_voltage_level(substation, mRID="vl_115", BaseVoltage="bv_115kV")
voltage_level, busbars, diameters = builder.add_buses(voltage_level, diameters=2)
builder.add_branch(voltage_level, diameter=1, side=1, node="line_1")
builder.add_branch(voltage_level, diameter=1, side=2, node=sourcebus(network, IEEE13))
builder.add_branch(voltage_level, diameter=2, side=1, node=sourcebus(network, IEEE13_ASSETS))
link_feeders(network, substation)
save_and_check(network, substation, "breaker_and_half.xml")

sub_breaker_and_half: feeders ['ieee13nodeckt', 'ieee13nodecktassets']
  21 breakers and disconnectors in the substation
  total load served: 6937.0 kW
